In [1]:
!pip install -U datasets huggingface_hub pillow

Defaulting to user installation because normal site-packages is not writeable
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
Using cached datasets-5.1.0-py3-none-any.whl (604 kB)
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ------------ --------------------------- 262.1/839.9 kB ? eta -:--:--
   ------------ --------------------------- 262.1/839.9 kB ? eta -:--:--
   ---------------------------------------- 839.9/839.9 kB 1.4 MB/s  0:00:00

  Attempting uninstall: huggingface_hub

    Found existing installation: huggingface_hub 0.36.2

   ---------------------------------------- 0/2 [huggingface_hub]
    Uninstalling huggingface_hub-0.36.2:
   ---------------------------------------- 0/2 [huggingface_hub]
      Successfully uninstalled huggingface_hub-0.36.2
   ---------------------------------------- 0/2 [huggingface_hub]
   ---------------------------------------- 0/2 [huggingface_hub]
   ---------------------------------------- 0/2 [hug

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.22.2 requires accelerate>=1.4.0, which is not installed.
sentence-transformers 6.1.0 requires huggingface-hub<2.0.0,>=1.3.0, but you have huggingface-hub 2.2.0 which is incompatible.
sentence-transformers 6.1.0 requires transformers<6.0.0,>=5.0.0, but you have transformers 4.57.1 which is incompatible.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.2.0 which is incompatible.


In [2]:
import os
from datasets import Dataset, Features, Value, Image
# from huggingface_hub import login

C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 2) Your image folder
IMG_DIR = "./iphone_imgs"  # change this

In [5]:
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

In [6]:
image_files

['./iphone_imgs\\image01.jpg',
 './iphone_imgs\\image02.jpg',
 './iphone_imgs\\image03.jpg',
 './iphone_imgs\\image04.jpg',
 './iphone_imgs\\image05.jpg']

In [8]:
# 3) Captions for images
captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

In [ ]:
# assert len(image_files) == len(captions), "Images and captions count must match!"

In [9]:
instruction = "Describe this iPhone product image in one sentence."

In [10]:
rows = []

In [11]:
for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [12]:
rows

[{'image': './iphone_imgs\\image01.jpg',
  'text': 'A white iPhone shown from the back and front on a white background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs\\image01.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A white iPhone shown from the back and front on a white background.'}]}]},
 {'image': './iphone_imgs\\image02.jpg',
  'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs\\image02.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.'}]}]},
 {'image': './iphone_imgs\

In [13]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

In [14]:
ds = Dataset.from_list(rows, features=features)

In [15]:
# If you want to store messages as real objects, easiest is to NOT force Features for messages.
ds = Dataset.from_list(rows)  # messages stored as nested objects

In [16]:
# 4) Make train/test splits (with only 5 images, keep test=1)
splits = ds.train_test_split(test_size=1, seed=3407)

In [19]:
# 5) Push to Hub
REPO_ID = "pratibha/iphone5_vlm_img"  # change this

In [18]:
# splits["train"].push_to_hub(REPO_ID, split="train")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 350.34ba/s]


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6ad33f76-0aed42032b2ff09c681421ef;3b15c0d5-28c8-4ef3-9776-0c95848b8e89)

Repository Not Found for url: https://huggingface.co/api/datasets/sunny199/iphone5_vlm_img/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
splits["train"].push_to_hub(REPO_ID, split="train")

In [ ]:
splits["test"].push_to_hub(REPO_ID, split="test")

In [ ]:
print("Uploaded:", REPO_ID)

In [20]:
from datasets import load_dataset

In [21]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharm\.cache\huggingface\hub\datasets--HuggingFaceM4--ChartQA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Program Files\Windows

In [22]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [23]:
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")

In [24]:
print(train_ds)

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})


In [25]:
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x19799BA52E0>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


In [26]:
"qwen2_vl_2b"
"qwen25_vl_3b"
"qwen25_omni_3b"
"medgemma_4b_vision"
"qwen25_vl_7b"
"qwen2_vl_7b"
"qwen25_omni_7b"
"llava15_7b"
"llava16_mistral_7b"

'llava16_mistral_7b'

In [27]:
# --- Install (Colab) ---
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.1.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 6.1.0 requires huggingface-hub<2.0.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 6.1.0 requires transformers<6.0.0,>=5.0.0, but you have transformers 4.57.1 which is incompatible.
ERROR: Ignored the following versions that require a different python version: 2025.3.2 Requires-Python >=3.9,<=3.12; 2025.3.4 Requires-Python >=3.9,<=3.12
ERROR: Could not find a version that satisfies the requirement triton (from versions: none)
ERROR: No matching distribution found for triton
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [30]:
# ==========================================================
# Universal Unsloth Vision Fine-tuning (Model + Dataset Dynamic)
# Supports: Qwen VL, LLaVA, Pixtral, MedGemma, Llama Vision
# Dataset selectable
# Trains on 150 row subset
# Runs for 2 Epochs
# ==========================================================

%pip install -U unsloth unsloth_zoo

import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
# ==========================================================
# 1️Model Registry
# ==========================================================
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

In [ ]:
# ==========================================================
# Dataset Registry (Vision-ready datasets)
# ==========================================================
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "sunny199/iphone5_vlm",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [ ]:
# ==========================================================
# Config
# ==========================================================
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "iphone_custom"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

In [ ]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg

        # Get model and dataset details from registry
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

        self.model = None
        self.tokenizer = None
        self.train_ds = None
        self.eval_ds = None
        self.trainer = None

    # -----------------------------
    # Load Model + Apply LoRA
    # -----------------------------
    def load_model(self):
        print("Loading model:", self.model_name)

        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        print("Model loaded and LoRA applied.")
        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):
        print("Loading dataset:", self.dataset_info["name"])

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        # Use small subset for demo/testing
        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(
            format_sample,
            remove_columns=raw.column_names
        )

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        print("Train samples:", len(self.train_ds))
        print("Eval samples:", len(self.eval_ds))

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):
        print("Building trainer...")

        FastVisionModel.for_training(self.model)

        training_args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",

            # Important for vision-language fine-tuning
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},

            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(
                self.model,
                self.tokenizer
            ),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=training_args,
        )

        print("Trainer ready.")
        return self

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self):
        print("Training started for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()
        print("Training completed.")
        return self

    # -----------------------------
    # Save Model
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)

        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)

        print("Model saved to:", self.cfg.save_dir)
        return self

    # -----------------------------
    # Quick Inference Test
    # -----------------------------
    def quick_infer(self, sample_index=0):
        print("Running quick inference...")

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": self.dataset_info["instruction"]},
                    {"type": "image"},
                ],
            }
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(
            self.tokenizer,
            skip_prompt=True
        )

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
            do_sample=True,
        )

        return self

    # -----------------------------
    # Full Pipeline Runner
    # -----------------------------
    def run(self):
        self.load_model()
        self.prepare_data()
        self.build_trainer()
        self.train()
        self.save()
        self.quick_infer()